In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


# BSM L04D — MFA w aplikacji mobilnej: TOTP, limity prób, replay i fallback

## Tryb pracy
To nie jest lab z Pythonem. Pracujesz głównie na projekcie **Android Studio / Kotlin**,
a notebook służy jako **formularz odpowiedzi**.

## Ciągłość z poprzednimi laboratoriami
- W `L01` modelowaliśmy zagrożenia i triadę AIC.
- W `L02` budowaliśmy intuicję kryptologiczną i pojęcie OTP.
- W `L03` analizowaliśmy hasła, tokeny i życie sekretów w aplikacji.
- W `L04` dokładamy **drugi składnik** i patrzymy, gdzie MFA realnie pomaga, a gdzie daje tylko pozorne bezpieczeństwo.

## Co robisz
1. Pobierasz projekt Kotlin `student/apps/lesson_d_app` przygotowany do laboratorium 4.
2. Uruchamiasz aplikację `Secret Lab MFA` i testy jednostkowe.
3. W kolejnych zadaniach analizujesz i poprawiasz klasy `TotpEngine`, `AttemptLimiter`, `MfaLoginGateway` oraz ścieżkę awaryjną.
4. Uzupełniasz pola formularza w tym notebooku.
5. Notebook sam składa `final_answer` dla każdego zadania.

W polu na górze wklejasz **link do projektu Kotlin**, nie link do notebooka.

## Link do projektu
Przed publikacją notebooka uzupełnij link poniżej:

- `TO_FILL_BEFORE_PUBLISH_WITH_PROJECT_ZIP`

## Konto testowe
- email: `student@lab.invalid`
- hasło: `student123!`
- seed TOTP: przygotowany w starterze projektu i provisioningu `otpauth://...`
- na ekranie MFA aplikacja pokazuje QR do zeskanowania w Google Authenticator / Aegis
- jeśli chcesz ręcznie, użyj URI zwracanego przez `LabMfaFixture.provisioningUri()`

## Ważne zasady
- Każde zadanie wysyłasz osobno.
- Jeśli zadanie wymaga logu albo wyniku testu, wpisz tylko potrzebny fragment.
- Nie przebudowuj całej aplikacji. Masz skupić się na aspektach bezpieczeństwa mechanizmu MFA.


In [ ]:
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    wyslij_odpowiedz(task_id, final_answer)


# D01 — Klasyfikacja metod MFA

Na podstawie wykładu 4 przypisz czterem metodom uwierzytelniania ich dominującą własność bezpieczeństwa.

Dla każdej pozycji wybierz jedną etykietę:
- `REPLAY_RESISTANT`,
- `PHISHING_RESISTANT`,
- `CHANNEL_WEAK`,
- `SERVER_SECRET_EXPOSED`.

Metody:
- SMS OTP,
- TOTP,
- S/KEY,
- U2F / passkey.


In [ ]:
#@title D01 — Formularz odpowiedzi
sms_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]
totp_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]
skey_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]
u2f_role = "" #@param ["", "REPLAY_RESISTANT", "PHISHING_RESISTANT", "CHANNEL_WEAK", "SERVER_SECRET_EXPOSED"]

final_answer = prepare_answer(
    "sms=" + sms_role,
    "totp=" + totp_role,
    "skey=" + skey_role,
    "u2f=" + u2f_role,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D01", final_answer)


# D02 — Provisioning sekretu TOTP

W starterze znajdź miejsce odpowiedzialne za zapis lub przekazanie sekretu TOTP.
Przejdź po kodzie od `SecureSessionStore.ensureProvisioned()` do ekranu MFA i ustal:
1. gdzie aplikacja przechowuje długoterminowy sekret,
2. czy sekret albo `otpauth://...` pojawia się w logach,
3. który plik lub klasa odpowiada za provisioning i który log wypisuje pełne `otpauth://...`.

Wpisz dokładnie:
- nazwę klasy lub pliku odpowiedzialnego za provisioning,
- typ magazynu użytego do zapisu sekretu,
- czy sekret wycieka do logów (`YES` albo `NO`),
- pierwsze 10 znaków URI użytego do enrolmentu.

Nie wpisuj całego sekretu Base32.


In [ ]:
#@title D02 — Formularz odpowiedzi
provisioning_place = "" #@param {type:"string"}
totp_secret_store = "" #@param {type:"string"}
secret_in_logs = "" #@param ["", "YES", "NO"]
issuer_or_uri_prefix = "" #@param {type:"string"}

final_answer = prepare_answer(
    "place=" + short_text(provisioning_place),
    "store=" + short_text(totp_secret_store),
    "log=" + secret_in_logs,
    "prefix=" + short_text(issuer_or_uri_prefix, limit=16),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D02", final_answer)


# D03 — Implementacja TOTP w Kotlinie

W projekcie pracujesz bezpośrednio na klasie `TotpEngine.kt`.
Jej celem jest poprawne wyliczenie i weryfikacja kodu 6-cyfrowego.

Wykonaj dokładnie tak:
1. Otwórz plik `student/apps/lesson_d_app/app/src/main/java/com/example/secretlab/mfa/TotpEngine.kt`.
2. Uzupełnij miejsca oznaczone `TODO(D03-...)`.
3. Uruchom testy `TotpEngineStudentTest`.
4. Odczytaj nazwę testu referencyjnego `matchesReferenceVectorAtKnownTimestamp` oraz wynik całego pakietu `TotpEngineStudentTest`.

W odpowiedzi wpisz:
- nazwę klasy implementującej TOTP,
- nazwę testu, który potwierdza zgodność z wektorem referencyjnym,
- czy cały pakiet testów przeszedł (`YES` albo `NO`),
- długość kodu OTP używaną w projekcie.


In [ ]:
#@title D03 — Formularz odpowiedzi
totp_engine_class = "" #@param {type:"string"}
totp_reference_test = "" #@param {type:"string"}
totp_tests_passed = "" #@param ["", "YES", "NO"]
otp_digits = "" #@param {type:"string"}

final_answer = prepare_answer(
    "engine=" + short_text(totp_engine_class),
    "test=" + short_text(totp_reference_test),
    "pass=" + totp_tests_passed,
    "digits=" + short_text(otp_digits, limit=8),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D03", final_answer)


# D04 — Okno czasowe i drift

TOTP zależy od czasu, więc system zwykle dopuszcza ograniczone okno tolerancji.
W starterze sprawdź w `TotpEngine(window = 1)`, jaki jest aktualny `window size` i jak wpływa on na bezpieczeństwo.

Wykonaj dokładnie tak:
1. Otwórz klasę `TotpEngine`.
2. Odczytaj liczbę akceptowanych slotów czasu po obu stronach aktualnego kroku.
3. Uruchom test poprawnego kodu z aktualnego slotu.
4. Uruchom test `acceptsCodeFromAdjacentTimeStepWhenWindowIsOne`.
5. Oceń, czy ustawione okno jest bardziej konserwatywne czy bardziej liberalne.

Wpisz dokładnie:
- rozmiar okna, np. `0`, `1`, `2`,
- czy kod z sąsiedniego slotu jest akceptowany (`YES` albo `NO`),
- ocenę polityki: `strict` albo `lenient`.


In [ ]:
#@title D04 — Formularz odpowiedzi
totp_window_size = "" #@param {type:"string"}
adjacent_slot_accepted = "" #@param ["", "YES", "NO"]
window_policy = "" #@param ["", "strict", "lenient"]

final_answer = prepare_answer(
    "window=" + short_text(totp_window_size, limit=8),
    "adj=" + adjacent_slot_accepted,
    "policy=" + window_policy,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D04", final_answer)


# D05 — Limity prób i blokada

Samo TOTP nie wystarczy, jeśli endpoint weryfikacji pozwala na nieograniczoną liczbę prób.
W starterze pracujesz na komponencie `AttemptLimiter` i dodajesz ograniczenie.

Wykonaj dokładnie tak:
1. Otwórz klasę `student/apps/lesson_d_app/app/src/main/java/com/example/secretlab/mfa/AttemptLimiter.kt`.
2. Uzupełnij miejsca oznaczone `TODO(D05-...)`.
3. Uruchom test `blocksAfterConfiguredNumberOfFailures` z `AttemptLimiterStudentTest`.
4. Odczytaj po ilu błędnych próbach następuje blokada i jak długo trwa cooldown.

W odpowiedzi wpisz:
- nazwę klasy limitera,
- maksymalną liczbę błędnych prób,
- długość cooldownu w sekundach,
- czy test blokady przechodzi (`YES` albo `NO`).


In [ ]:
#@title D05 — Formularz odpowiedzi
attempt_limiter_class = "" #@param {type:"string"}
max_bad_attempts = "" #@param {type:"string"}
cooldown_seconds = "" #@param {type:"string"}
limiter_test_passed = "" #@param ["", "YES", "NO"]

final_answer = prepare_answer(
    "limiter=" + short_text(attempt_limiter_class),
    "max=" + short_text(max_bad_attempts, limit=8),
    "cool=" + short_text(cooldown_seconds, limit=8),
    "pass=" + limiter_test_passed,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D05", final_answer)


# D06 — Replay i związanie kodu z próbą logowania

Kod TOTP nie powinien być traktowany jako samodzielny bilet dostępu oderwany od kontekstu.
W starterze sprawdź to na klasie `MfaLoginGateway`, czyli czy poprawny kod można wykorzystać ponownie albo w innej sesji logowania.

Wykonaj dokładnie tak:
1. Otwórz klasę `student/apps/lesson_d_app/app/src/main/java/com/example/secretlab/mfa/MfaLoginGateway.kt`.
2. Sprawdź, czy weryfikacja korzysta z dokładnego identyfikatora bieżącej próby logowania `challengeId`.
3. Uruchom testy `rejectsOtpForDifferentChallengeId` oraz `rejectsReplayOfCodeFromAlreadyAcceptedTimeStep`.
4. Oceń, czy implementacja jest odporna na prosty replay w obrębie ważnego okna czasu.

Wpisz dokładnie:
- nazwę parametru albo artefaktu wiążącego MFA z sesją logowania,
- czy ten sam kod da się użyć ponownie (`YES` albo `NO`),
- ocenę: `bound` albo `unbound`.


In [ ]:
#@title D06 — Formularz odpowiedzi
session_binding_artifact = "" #@param {type:"string"}
otp_reusable = "" #@param ["", "YES", "NO"]
binding_assessment = "" #@param ["", "bound", "unbound"]

final_answer = prepare_answer(
    "bind=" + short_text(session_binding_artifact),
    "reuse=" + otp_reusable,
    "mode=" + binding_assessment,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D06", final_answer)


# D07 — Słaby fallback obchodzi mocne MFA

W wykładzie 4 kluczowa teza brzmi: przeciwnik często nie atakuje głównej ścieżki MFA,
tylko słabszy fallback.

W starterze znajdź awaryjną ścieżkę odzyskiwania dostępu i oceń ją.

Wykonaj dokładnie tak:
1. Otwórz ekran `Fallback recovery` w `MainActivity.kt` oraz metodę `fallbackRecover(...)` w `MfaLoginGateway.kt`.
2. Ustal, jaki mechanizm jest tam użyty. W tym starterze to `email code`.
3. Odczytaj, czy ten mechanizm omija główny etap TOTP.
4. Nadaj mu ocenę ryzyka z perspektywy wykładu 4.

Wpisz dokładnie:
- nazwę mechanizmu fallback,
- czy fallback omija TOTP (`YES` albo `NO`),
- ocenę: `weaker_than_main` albo `comparable_to_main`.


In [ ]:
#@title D07 — Formularz odpowiedzi
fallback_mechanism = "" #@param {type:"string"}
fallback_bypasses_totp = "" #@param ["", "YES", "NO"]
fallback_assessment = "" #@param ["", "weaker_than_main", "comparable_to_main"]

final_answer = prepare_answer(
    "fallback=" + short_text(fallback_mechanism),
    "bypass=" + fallback_bypasses_totp,
    "risk=" + fallback_assessment,
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D07", final_answer)


# D08 — Ocena końcowa architektury logowania

Po wykonaniu poprzednich zadań oceń gotową architekturę `lesson_d_app`.
Nie chodzi o marketingowe hasło `mamy MFA`, tylko o precyzyjne wskazanie, jaki poziom ochrony rzeczywiście uzyskano.

Wpisz krótką ocenę końcową systemu po poprawkach:
- czy główna ścieżka jest odporna na prosty replay (`YES` albo `NO`),
- czy główna ścieżka jest phishing-resistant (`YES` albo `NO`),
- który element nadal jest najsłabszy: `fallback`, `provisioning`, `device_compromise`, `logging`, `time_window`,
- jednozdaniową rekomendację kolejnego kroku architektonicznego.


In [ ]:
#@title D08 — Formularz odpowiedzi
replay_resistant = "" #@param ["", "YES", "NO"]
phishing_resistant = "" #@param ["", "YES", "NO"]
weakest_element = "" #@param ["", "fallback", "provisioning", "device_compromise", "logging", "time_window"]
next_arch_step = "" #@param {type:"string"}

final_answer = prepare_answer(
    "replay=" + replay_resistant,
    "phish=" + phishing_resistant,
    "weak=" + weakest_element,
    "next=" + short_text(next_arch_step, limit=80),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("D08", final_answer)
